Check 1 — Load the feature dataset (baseline)

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/features/BTC_features.csv")
df.head()


,date,open,high,low,close,volume,ticker,market_cap,log_ret_1d,log_ret_5d,log_ret_10d,vol_7d,vol_30d,risk_adj_ret_1d,vol_ratio_7d_30d,drawdown_30d
0,2020-01-31,9508.313477,9521.706055,9230.776367,9350.529297,29432489719,BTC-USD,NaN,-0.016805,0.084039,0.066849,0.024991,0.028110,-0.597831,0.889034,-0.016665
1,2020-02-01,9346.357422,9439.323242,9313.239258,9392.875000,25922656496,BTC-USD,NaN,0.004518,0.052797,0.078829,0.023294,0.027147,0.166442,0.858044,-0.012211
2,2020-02-02,9389.820312,9468.797852,9217.824219,9344.365234,30835736946,BTC-USD,NaN,-0.005178,-0.001521,0.105766,0.024041,0.026178,-0.197799,0.918398,-0.017313
3,2020-02-03,9344.683594,9540.372070,9248.633789,9293.521484,30934096509,BTC-USD,NaN,-0.005456,-0.002483,0.095692,0.022204,0.026292,-0.207515,0.844516,-0.022660
4,2020-02-04,9292.841797,9331.265625,9112.811523,9180.962891,29893183716,BTC-USD,NaN,-0.012185,-0.035106,0.092735,0.012202,0.026507,-0.459713,0.460328,-0.034497


In [2]:
print(df.shape)
print(df.columns)

(2201, 16)
Index(['date', 'open', 'high', 'low', 'close', 'volume', 'ticker',
       'market_cap', 'log_ret_1d', 'log_ret_5d', 'log_ret_10d', 'vol_7d',
       'vol_30d', 'risk_adj_ret_1d', 'vol_ratio_7d_30d', 'drawdown_30d'],
      dtype='str')


Check 2 — Missing values (CRITICAL)

In [3]:
print(df.isna().sum())


date                   0
open                   0
high                   0
low                    0
close                  0
volume                 0
ticker                 0
market_cap          1837
log_ret_1d             0
log_ret_5d             0
log_ret_10d            0
vol_7d                 0
vol_30d                0
risk_adj_ret_1d        0
vol_ratio_7d_30d       0
drawdown_30d           0
dtype: int64


Check 3 — Date ordering (CRITICAL for time series): Any out-of-order row causes data leakage

In [4]:
df["date"] = pd.to_datetime(df["date"])

print("Is monotonic increasing:", df["date"].is_monotonic_increasing)
print("First date:", df["date"].iloc[0])
print("Last date:", df["date"].iloc[-1])


Is monotonic increasing: True
First date: 2020-01-31 00:00:00
Last date: 2026-02-09 00:00:00


Check 4 — Duplicate dates (CRITICAL)

In [5]:
duplicates = df["date"].duplicated().sum()
print("Duplicate dates:", duplicates)


Duplicate dates: 0


Check 5 — Return sanity check (IMPORTANT)

In [6]:
df[["log_ret_1d", "log_ret_5d", "log_ret_10d"]].describe(
    percentiles=[0.01, 0.05, 0.95, 0.99]
)


,log_ret_1d,log_ret_5d,log_ret_10d
count,2201.000000,2201.000000,2201.000000
mean,0.000913,0.004601,0.009651
std,0.032536,0.071530,0.102793
min,-0.464730,-0.583591,-0.598419
1%,-0.089140,-0.191097,-0.282479
5%,-0.048300,-0.107791,-0.139126
95%,0.048825,0.120722,0.180370
99%,0.094901,0.180709,0.273974
max,0.171821,0.230365,0.399452


Check 6 — Volatility sanity check

Volatility should: always be ≥ 0. move smoothly over time

In [ ]:
print(df[["vol_7d", "vol_30d"]].describe())

# short-term volatility is noisier

# long-term volatility smoother

            vol_7d      vol_30d
count  2201.000000  2201.000000
mean      0.027727     0.029458
std       0.016756     0.013122
min       0.003447     0.008926
25%       0.017374     0.021238
50%       0.024276     0.027335
75%       0.034193     0.034808
max       0.190284     0.103649


In [ ]:
# Also check for negatives: Important for regime modeling.

print((df["vol_7d"] < 0).sum(), (df["vol_30d"] < 0).sum())


0 0


Check 7 — New feature sanity (VERY IMPORTANT)

In [ ]:
df[[
    "risk_adj_ret_1d",
    "vol_ratio_7d_30d",
    "drawdown_30d"
]].describe(percentiles=[0.01, 0.05, 0.95, 0.99])

# risk_adj_ret_1d: can be >1 or <-1 (that’s fine)
# vol_ratio_7d_30d: centered near ~1,spikes above 1 during turbulence

# drawdown_30d: ≤ 0, 0 means recent peak

,risk_adj_ret_1d,vol_ratio_7d_30d,drawdown_30d
count,2201.000000,2201.000000,2201.000000
mean,0.039361,0.944539,-0.084361
std,1.023421,0.339259,0.084803
min,-5.275672,0.167682,-0.518617
1%,-2.804578,0.291970,-0.376147
5%,-1.648343,0.448422,-0.256738
95%,1.725038,1.542900,0.000000
99%,2.844728,1.837601,0.000000
max,4.800255,2.040111,0.000000


Check 7 — New feature sanity (THIS IS A BIG WIN)
risk_adj_ret_1d

Mean near 0, Heavy tails, Symmetric-ish, This is exactly what a risk-normalized signal should look like.

vol_ratio_7d_30d

Mean < 1, Spikes above 1, Wide distribution, This will separate calm vs unstable regimes extremely well.

drawdown_30d

Max = 0, Min ≈ −52%, Distribution makes sense, This confirms drawdown is implemented correctly and psychologically meaningful.

In [10]:
print("Max drawdown value:", df["drawdown_30d"].max())


Max drawdown value: 0.0


Check 8 — Correlation overview (OPTIONAL but insightful)

In [14]:
features = [
    "log_ret_1d", "log_ret_5d", "log_ret_10d",
    "vol_7d", "vol_30d",
    "risk_adj_ret_1d", "vol_ratio_7d_30d", "drawdown_30d"
]

print(df[features].corr())


                  log_ret_1d  log_ret_5d  log_ret_10d    vol_7d   vol_30d  \
log_ret_1d          1.000000    0.466529     0.322568 -0.041534 -0.006760   
log_ret_5d          0.466529    1.000000     0.720697 -0.181153 -0.038159   
log_ret_10d         0.322568    0.720697     1.000000 -0.253384 -0.077626   
vol_7d             -0.041534   -0.181153    -0.253384  1.000000  0.677237   
vol_30d            -0.006760   -0.038159    -0.077626  0.677237  1.000000   
risk_adj_ret_1d     0.916421    0.434487     0.311541 -0.026419 -0.018356   
vol_ratio_7d_30d   -0.020298   -0.102261    -0.136928  0.627949 -0.021970   
drawdown_30d        0.264017    0.529092     0.663676 -0.485777 -0.524247   

                  risk_adj_ret_1d  vol_ratio_7d_30d  drawdown_30d  
log_ret_1d               0.916421         -0.020298      0.264017  
log_ret_5d               0.434487         -0.102261      0.529092  
log_ret_10d              0.311541         -0.136928      0.663676  
vol_7d                  -0.026419 

Returns vs returns

log_ret_5d ↔ log_ret_10d ≈ 0.72, expected (momentum overlap), not redundant

Volatility structure

vol_7d ↔ vol_30d ≈ 0.68, strong but not identical, good for regime detection

Risk-adjusted return

risk_adj_ret_1d ↔ log_ret_1d ≈ 0.92, expected (it’s normalized return), still useful because scale matters for ML

Drawdown relationships

Strongly correlated with longer returns, Strongly negatively correlated with volatility, This is very consistent with financial intuition.

Nothing here suggests feature leakage or redundancy problems.

Check 9 — Visual gap sanity (OPTIONAL)

In [13]:
df["date"].diff().dt.days.value_counts().sort_index()


date
1.0    2199
2.0       1
Name: count, dtype: int64

This is weekend behavior, expected, harmless